# Figure 5: compare PPO checkpoints

- Loads the saved 1k–50k evaluation CSV for Normalized, Fract, CNN Mesh, and Mesh PPO.
- Evaluates only missing checkpoints; plotting a complete CSV does not require checkpoint files.
- Runs deterministic evaluation over 30 rounds with 10 parallel simulations and tumour IDs 1–10, using 24h availability without treatment delays.
- Saves the evaluation CSV and a learning-curve figure.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))

import gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from stable_baselines3 import PPO

PROJECT_ROOT = next((candidate.resolve() for parent in (Path.cwd(), *Path.cwd().parents) for candidate in (parent, parent / "report_files") if (candidate / "train_normalized_PPO.py").is_file() and (candidate / "TumorGrowthEnv/tumor_growth/__init__.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the PPO project directory")
for local_dir in (PROJECT_ROOT, PROJECT_ROOT / "TumorGrowthEnv"):
    if str(local_dir) not in sys.path:
        sys.path.insert(0, str(local_dir))
import cnn_mesh_policy
import tumor_growth

if torch.cuda.is_available():
    gpu_capability = torch.cuda.get_device_capability(0)
    gpu_architecture = f"sm_{gpu_capability[0]}{gpu_capability[1]}"
    if gpu_architecture in torch.cuda.get_arch_list():
        DEVICE = torch.device("cuda:0")
        print(f"Evaluation device: {DEVICE} ({torch.cuda.get_device_name(DEVICE)})")
    else:
        DEVICE = torch.device("cpu")
        print(
            f"PyTorch {torch.__version__} does not support {gpu_architecture} "
            f"for {torch.cuda.get_device_name(0)}; using CPU for evaluation."
        )
else:
    DEVICE = torch.device("cpu")
    print("CUDA is unavailable; using CPU for evaluation.")

PARALLEL_RUNS = 10
ROUNDS = 30
TUMOUR_IDS = list(range(1, 11))
DETERMINISTIC = True
START_STEP = 1_000
END_STEP = 50_000
STEP_SIZE = 1_000
RESULTS_DIR = PROJECT_ROOT / "baselines" / "results"
CACHE_SOURCES = [
    PROJECT_ROOT / "other_utilities" / "data" / f"normalized_fract_checkpoint_evaluation_{ROUNDS}x{PARALLEL_RUNS}_deterministic.csv",
    PROJECT_ROOT / "other_utilities" / "data" / f"cnn_mesh_checkpoint_evaluation_{ROUNDS}x{PARALLEL_RUNS}_deterministic.csv",
    RESULTS_DIR / f"fig05_checkpoint_evaluation_normalized_fract_cnn_mesh_1k_to_50k_{ROUNDS}x{PARALLEL_RUNS}.csv",
]
CACHE_PATH = RESULTS_DIR / f"fig05_checkpoint_evaluation_four_models_1k_to_50k_{ROUNDS}x{PARALLEL_RUNS}.csv"
MODEL_COLORS = {
    "Normalized PPO": "#E68613",
    "Fract PPO": "#7B2CBF",
    "CNN Mesh PPO": "#2CA25F",
    "Mesh PPO": "#1F77B4",
}


In [ ]:
MODEL_DIRS = {
    "Normalized PPO": RESULTS_DIR / "models/normalized_ppo",
    "Fract PPO": RESULTS_DIR / "models/fract_ppo",
    "CNN Mesh PPO": RESULTS_DIR / "models/cnn_mesh_ppo",
    "Mesh PPO": RESULTS_DIR / "models/mesh_ppo",
}
ENVIRONMENT_IDS = {
    "Normalized PPO": "tumor_growth:tumor-growth-normalized-ppo-v1",
    "Fract PPO": "tumor_growth:tumor-growth-fract-ppo-v1",
    "CNN Mesh PPO": "tumor_growth:tumor-growth-cnn-mesh-ppo-v1",
    "Mesh PPO": "tumor_growth:tumor-growth-mesh-ppo-v1",
}
EXPECTED_STEPS = set(range(START_STEP, END_STEP + STEP_SIZE, STEP_SIZE))


def available_checkpoints(model_dir):
    checkpoints = {}
    if not model_dir.is_dir():
        raise FileNotFoundError(model_dir)
    for checkpoint_path in model_dir.rglob("ppo_*_steps.zip"):
        step_text = checkpoint_path.stem.removeprefix("ppo_").removesuffix("_steps")
        if not step_text.isdigit():
            continue
        step = int(step_text)
        if START_STEP <= step <= END_STEP and step % STEP_SIZE == 0:
            if step in checkpoints:
                raise RuntimeError(
                    f"Duplicate checkpoint for {step} steps: "
                    f"{checkpoints[step]} and {checkpoint_path}"
                )
            checkpoints[step] = checkpoint_path
    return checkpoints


print("Checkpoint files are required only for evaluation rows missing from the saved CSV.")


In [ ]:
RESULT_COLUMNS = [
    "model", "training_steps", "training_steps_k",
    "mean_leftover_cells", "std_between_round_means",
    "standard_error", "ci95", "rounds", "parallel_runs",
    "tumour_ids", "deterministic",
]

cache_frames = []
for cache_source in [*CACHE_SOURCES, CACHE_PATH]:
    if cache_source.is_file():
        frame = pd.read_csv(cache_source)
        missing_columns = set(RESULT_COLUMNS) - set(frame.columns)
        if missing_columns:
            raise ValueError(f"{cache_source} is missing columns: {sorted(missing_columns)}")
        cache_frames.append(frame[RESULT_COLUMNS])
        print(f"Loaded saved evaluations from {cache_source}")

if cache_frames:
    results_df = pd.concat(cache_frames, ignore_index=True)
else:
    results_df = pd.DataFrame(columns=RESULT_COLUMNS)

if not results_df.empty:
    results_df["training_steps"] = results_df["training_steps"].astype(int)
    compatible = (
        results_df["model"].isin(MODEL_DIRS)
        & results_df["training_steps"].between(START_STEP, END_STEP)
        & results_df["rounds"].eq(ROUNDS)
        & results_df["parallel_runs"].eq(PARALLEL_RUNS)
        & results_df["tumour_ids"].eq(len(TUMOUR_IDS))
        & results_df["deterministic"].astype(str).str.lower().eq(str(DETERMINISTIC).lower())
    )
    ignored_rows = int((~compatible).sum())
    if ignored_rows:
        print(f"Ignored {ignored_rows} cached rows outside the requested range or settings")
    results_df = results_df.loc[compatible].drop_duplicates(
        ["model", "training_steps"], keep="last"
    )


completed = set(zip(results_df.get("model", []), results_df.get("training_steps", [])))
missing_evaluations = {
    model_name: sorted(EXPECTED_STEPS - {step for model, step in completed if model == model_name})
    for model_name in MODEL_DIRS
}
checkpoint_maps = {}
for model_name, missing_steps in missing_evaluations.items():
    if not missing_steps:
        continue
    checkpoints = available_checkpoints(MODEL_DIRS[model_name])
    unavailable = sorted(set(missing_steps) - set(checkpoints))
    if unavailable:
        raise FileNotFoundError(
            f"Missing checkpoints needed to evaluate {model_name}: "
            + ", ".join(f"{step // 1000}k" for step in unavailable)
        )
    checkpoint_maps[model_name] = {step: checkpoints[step] for step in missing_steps}
if not checkpoint_maps:
    print("All checkpoint evaluations are present in the saved CSV.")
for model_name in MODEL_DIRS:
    for step, checkpoint_path in sorted(checkpoint_maps.get(model_name, {}).items()):
        print(f"Evaluating {model_name} at {step // 1000}k: {checkpoint_path}")
        env = gym.make(
            ENVIRONMENT_IDS[model_name],
            tumors_list=TUMOUR_IDS,
            parallel_runs=PARALLEL_RUNS,
            with_time=True,
            dosing_hours=24,
            treatment_delay=False,
        )
        predictor = PPO.load(str(checkpoint_path), device=DEVICE)
        print(f"Policy device: {predictor.policy.device}")

        round_means = []
        try:
            for _ in range(ROUNDS):
                observation = env.reset()
                if isinstance(observation, tuple):
                    observation = observation[0]
                done = False
                while not done:
                    action, _ = predictor.predict(observation, deterministic=DETERMINISTIC)
                    result = env.step(action)
                    if len(result) == 4:
                        observation, _, done, _ = result
                    else:
                        observation, _, terminated, truncated, _ = result
                        done = terminated or truncated
                cells = np.asarray(env.unwrapped.tumor_cells, dtype=np.float64).ravel()
                round_means.append(cells.mean())
        finally:
            env.close()

        round_means = np.asarray(round_means, dtype=np.float64)
        round_std = round_means.std(ddof=1)
        standard_error = round_std / np.sqrt(ROUNDS)
        row = pd.DataFrame([{
            "model": model_name,
            "training_steps": step,
            "training_steps_k": step / 1000,
            "mean_leftover_cells": round_means.mean(),
            "std_between_round_means": round_std,
            "standard_error": standard_error,
            "ci95": 1.96 * standard_error,
            "rounds": ROUNDS,
            "parallel_runs": PARALLEL_RUNS,
            "tumour_ids": len(TUMOUR_IDS),
            "deterministic": DETERMINISTIC,
        }])
        results_df = pd.concat([results_df, row], ignore_index=True)
        results_df = results_df.sort_values(["model", "training_steps"])
        temporary_cache = CACHE_PATH.with_suffix(".tmp.csv")
        results_df.to_csv(temporary_cache, index=False)
        temporary_cache.replace(CACHE_PATH)
        completed.add((model_name, step))

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results_df = results_df.sort_values(["model", "training_steps"]).reset_index(drop=True)
temporary_cache = CACHE_PATH.with_suffix(".tmp.csv")
results_df.to_csv(temporary_cache, index=False)
temporary_cache.replace(CACHE_PATH)
print(f"Saved combined evaluation data to {CACHE_PATH}")
display(results_df.round(3))


In [ ]:
plot_data = results_df[
    results_df["model"].isin(MODEL_COLORS)
    & results_df["training_steps"].between(START_STEP, END_STEP)
].copy()
for model_name in MODEL_COLORS:
    available_steps = set(plot_data.loc[plot_data["model"].eq(model_name), "training_steps"])
    missing_steps = sorted(EXPECTED_STEPS - available_steps)
    if missing_steps:
        raise RuntimeError(
            f"Incomplete evaluations for {model_name}: "
            + ", ".join(f"{step // 1000}k" for step in missing_steps)
        )

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(
    1, len(MODEL_COLORS), figsize=(18, 5.5), sharex=True, sharey=True,
)
for ax, (model_name, color) in zip(axes, MODEL_COLORS.items()):
    curve = plot_data[plot_data["model"].eq(model_name)].sort_values("training_steps")
    x = curve["training_steps_k"].to_numpy(dtype=float)
    mean = curve["mean_leftover_cells"].to_numpy(dtype=float)
    ci95 = curve["ci95"].to_numpy(dtype=float)
    ax.plot(x, mean, color=color, marker="o", markersize=4, linewidth=2.4)
    ax.fill_between(x, mean - ci95, mean + ci95, color=color, alpha=0.16)
    ax.set_title(model_name, color=color, fontweight="bold")
    ax.set_xlim(START_STEP / 1000, END_STEP / 1000)
    ax.set_xticks([1, 10, 20, 30, 40, 50])

fig.supxlabel("Training steps (thousands)")
fig.supylabel("Mean leftover tumour cells")
fig.suptitle("Deterministic PPO checkpoint comparison", fontsize=15, fontweight="bold")
fig.tight_layout()

plot_path = RESULTS_DIR / "fig05_four_model_learning_curves_1k_to_50k.png"
fig.savefig(plot_path, dpi=300, bbox_inches="tight")
print(f"Saved small-multiples plot to {plot_path}")
plt.show()
